# 🌾 AgroMAE: Modelado Auto-Supervisado de Imágenes Satelitales para Uso de Suelos
## Inteligencia Artificial II — Universidad Industrial de Santander

**Integrantes:**
* Andrés Felipe Alfonso Viviescas
* Andrés Felipe Higuera Grateron

---

### 🎯 Objetivos de este Cuaderno
1. **Pre-entrenamiento Auto-Supervisado (Fase 1):** Implementar un **Masked Autoencoder (MAE)** adaptado a imágenes multiespectrales de 13 bandas (Sentinel-2).
2. **Enmascaramiento Aleatorio del 75%:** Obligar a la red a reconstruir los parches ocultos mediante una función de pérdida espectral (MSE Loss).
3. **Sistema de Checkpoint Profesional:** Implementación con guardado en Google Drive en dos niveles (*Intra-Epoch* y *Epoch-Level*) con persistencia de pesos, momentos del optimizador AdamW y *GradScaler* para tolerancia a desconexiones de Colab.
4. **Aceleración para GPU NVIDIA L4:** Precisión mixta nativa (`bfloat16`/`float16`), `TF32` y *FlashAttention* (`scaled_dot_product_attention`).

## 1. ⚙️ Diagnóstico de Hardware y Configuración de Aceleración (GPU L4)

In [ ]:
import os
import sys
import time
import math
import shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# 1. Verificar GPU asignada
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📌 Dispositivo de cómputo: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🚀 GPU Detectada: {gpu_name} con {total_vram:.2f} GB de VRAM")
    
    # Activar optimizaciones para Tensor Cores de NVIDIA L4 (Ada Lovelace)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    
    # Soporte de precisión mixta: bfloat16 en arquitecturas modernas
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"⚡ Precisión Mixta seleccionada: {AMP_DTYPE}")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU. Por favor ve a Entorno de Ejecución > Cambiar tipo de entorno y selecciona GPU L4.")
    AMP_DTYPE = torch.float32

## 2. 📁 Montaje de Google Drive (Almacenamiento Persistente de Checkpoints)

In [ ]:
# Montar Google Drive para persistencia de Checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/AgroMAE_Proyecto"
except Exception as e:
    print("ℹ️ No se está ejecutando en Colab o se canceló el montaje. Usando almacenamiento local.")
    DRIVE_DIR = "./AgroMAE_Local"

CHECKPOINT_DIR = os.path.join(DRIVE_DIR, "checkpoints")
LOGS_DIR = os.path.join(DRIVE_DIR, "logs")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print(f"📂 Directorio de Checkpoints configurado en: {CHECKPOINT_DIR}")

## 3. 💾 Gestor de Checkpoints Profesional (`CheckpointManager`)
Este módulo atiende la recomendación del docente. Guarda:
* `model_state_dict`: Pesos del Encoder y Decoder.
* `optimizer_state_dict`: Momentos de inercia ($m_t, v_t$) del optimizador AdamW.
* `scheduler_state_dict`: Tasa de aprendizaje exacta (Cosine Decay con Warmup).
* `scaler_state_dict`: Estado del `GradScaler` de precisión mixta.
* Guardado *Intra-Epoch* (cada $N$ pasos) para tolerar desconexiones inesperadas de Colab.

In [ ]:
class CheckpointManager:
    def __init__(self, checkpoint_dir, max_to_keep=2):
        self.checkpoint_dir = checkpoint_dir
        self.max_to_keep = max_to_keep
        self.last_path = os.path.join(checkpoint_dir, "checkpoint_last.pth")
        self.best_path = os.path.join(checkpoint_dir, "checkpoint_best.pth")
        self.history = []

    def save(self, epoch, global_step, model, optimizer, scheduler=None, scaler=None, loss=None, is_best=False):
        state = {
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'loss': loss
        }
        
        # Guardar último estado (para reanudar tras desconexión)
        torch.save(state, self.last_path)
        
        # Guardar mejor modelo histórico
        if is_best:
            torch.save(state, self.best_path)
            print(f"⭐ [RECORD] Mejor modelo actualizado con Loss: {loss:.4f}")

    def load_latest(self, model, optimizer=None, scheduler=None, scaler=None, device="cuda"):
        if not os.path.exists(self.last_path):
            print("🚀 No se encontró checkpoint previo. Iniciando entrenamiento desde cero.")
            return 0, 0, float("inf")

        print(f"🔄 Reanudando desde checkpoint: {self.last_path}")
        ckpt = torch.load(self.last_path, map_location=device)
        
        model.load_state_dict(ckpt['model_state_dict'])
        if optimizer and ckpt.get('optimizer_state_dict'):
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        start_epoch = ckpt.get('epoch', 0)
        start_step = ckpt.get('global_step', 0)
        best_loss = ckpt.get('loss', float("inf"))

        print(f"✅ Estado reanudado exitosamente en Época {start_epoch + 1}, Paso global {start_step}.")
        return start_epoch + 1, start_step, best_loss

## 4. 🛰️ Descarga y Carga del Dataset Multiespectral (Sentinel-2, 13 Bandas)
Para garantizar velocidad y reproducibilidad en la máquina virtual, descargamos el dataset **EuroSAT Multiespectral** (13 bandas) directamente en el disco SSD local `/content/`.

In [ ]:
LOCAL_DATA_DIR = "/content/eurosat_ms"
ZIP_PATH = "/content/EuroSATallBands.zip"

# Descargar EuroSAT Multiespectral si no existe en el disco local
if not os.path.exists(LOCAL_DATA_DIR):
    print("📥 Descargando EuroSAT Multiespectral (Sentinel-2, 13 bandas, ~2 GB)...")
    !wget -q --show-progress -O /content/EuroSATallBands.zip https://zenodo.org/records/7745585/files/EuroSATallBands.zip
    print("📦 Descomprimiendo en disco SSD de Colab...")
    !unzip -q /content/EuroSATallBands.zip -d /content/eurosat_ms
    print("✅ Dataset descomprimido y listo para entrenamiento.")
else:
    print("✅ Dataset ya disponible en el disco local.")

### Dataset y DataLoader Multiespectral con Normalización por Bandas de Sentinel-2
Sentinel-2 tiene 13 bandas con valores de reflectancia de superficie (típicamente de 0 a 10,000). Normalizamos dividiendo por 10,000 para llevar los valores al rango físico $[0, 1]$.

In [ ]:
import glob
try:
    import rasterio
except ImportError:
    !pip install -q rasterio
    import rasterio

class Sentinel2Dataset(Dataset):
    """
    Dataset para imágenes satelitales multiespectrales (13 bandas de Sentinel-2).
    Carga archivos .tif de 64x64 píxeles con 13 canales.
    """
    def __init__(self, root_dir, transform=None):
        self.file_list = glob.glob(os.path.join(root_dir, "**", "*.tif"), recursive=True)
        self.transform = transform
        
        # Extraer clases a partir de los nombres de carpetas
        classes = sorted(list(set([os.path.basename(os.path.dirname(f)) for f in self.file_list])))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        print(f"📊 Total de imágenes multiespectrales encontradas: {len(self.file_list)}")
        print(f"🏷️ Clases ({len(classes)}): {classes}")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        with rasterio.open(file_path) as src:
            # Leer 13 bandas: (13, H, W)
            img = src.read().astype(np.float32)
        
        # Normalización radiométrica de Sentinel-2 (Top of Atmosphere reflectance factor = 10000)
        img = np.clip(img / 10000.0, 0.0, 1.0)
        tensor = torch.from_numpy(img)
        
        # Aumentación básica (Flips aleatorios espaciales)
        if self.transform:
            if torch.rand(1) > 0.5:
                tensor = torch.flip(tensor, dims=[-1])  # Horizontal
            if torch.rand(1) > 0.5:
                tensor = torch.flip(tensor, dims=[-2])  # Vertical
                
        cls_name = os.path.basename(os.path.dirname(file_path))
        label = self.class_to_idx.get(cls_name, 0)
        return tensor, label

# Instanciar Dataset y DataLoaders
full_dataset = Sentinel2Dataset(root_dir=LOCAL_DATA_DIR, transform=True)
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# Batch size optimizado para GPU L4 con 24 GB de VRAM
BATCH_SIZE = 128
NUM_WORKERS = 4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"✅ DataLoaders listos: {len(train_loader)} batches de entrenamiento (Batch size = {BATCH_SIZE})")

## 5. 🔬 Arquitectura AgroMAE (Masked Autoencoder Multiespectral)

### Componentes Clave:
1. **Embedding por Parches 2D:** Divide imágenes de $64\times 64$ en parches de $8\times 8$, generando $N = 64$ parches con $13$ canales cada uno.
2. **Algoritmo de Enmascaramiento Aleatorio (75%):** Selecciona aleatoriamente el $25\%$ de los parches para el *Encoder* y descarta el $75\%$ restante durante la compresión latente.
3. **Position Embedding Seno-Coseno 2D:** Incorpora geometría espacial fija sin parámetros aprendibles para conservar invariancia traslacional.
4. **Encoder ViT:** Bloques Transformer acelerados mediante `torch.nn.functional.scaled_dot_product_attention`.
5. **Decoder Ligero:** Reensambla los parches procesados con tokens de máscara aprendibles y proyecta a los $8\times 8 \times 13 = 832$ píxeles por parche.

In [ ]:
def get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False):
    """
    Generador de embeddings posicionales 2D basados en senos y cosenos (He et al., 2022).
    """
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)  # (2, grid_size, grid_size)
    grid = np.stack(grid, axis=0)
    grid = grid.reshape([2, 1, grid_size, grid_size])
    
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed

def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])
    return np.concatenate([emb_h, emb_w], axis=1)

def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega
    pos = pos.reshape(-1)
    out = np.einsum('m,d->md', pos, omega)
    emb_sin = np.sin(out)
    emb_cos = np.cos(out)
    return np.concatenate([emb_sin, emb_cos], axis=1)

class TransformerBlock(nn.Module):
    """Bloque Transformer con FlashAttention nativo"""
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim)
        )

    def forward(self, x):
        B, N, C = x.shape
        # Self-Attention con PyTorch 2.0+ Scaled Dot Product Attention
        x_norm = self.norm1(x)
        qkv = self.qkv(x_norm).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn_out = F.scaled_dot_product_attention(q, k, v)
        attn_out = attn_out.transpose(1, 2).reshape(B, N, C)
        x = x + self.proj(attn_out)
        
        # MLP
        x = x + self.mlp(self.norm2(x))
        return x

### Definición del Modelo Completo `AgroMAE`

In [ ]:
class AgroMAE(nn.Module):
    """
    Masked Autoencoder adaptado para Teledetección Multiespectral (Sentinel-2).
    """
    def __init__(self, img_size=64, patch_size=8, in_chans=13,
                 embed_dim=256, depth=6, num_heads=8,
                 decoder_embed_dim=128, decoder_depth=3, decoder_num_heads=4,
                 mask_ratio=0.75):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.mask_ratio = mask_ratio
        
        # 1. Patch Embedding Multiespectral
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        
        # 2. Encoder ViT
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim), requires_grad=False)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        
        # 3. Decoder ViT Ligero
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, decoder_embed_dim), requires_grad=False)
        self.decoder_blocks = nn.ModuleList([
            TransformerBlock(decoder_embed_dim, decoder_num_heads) for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size * patch_size * in_chans, bias=True)
        
        self.initialize_weights()

    def initialize_weights(self):
        # Inicializar embeddings posicionales fijos 2D sincos
        pos_embed = get_2d_sincos_pos_embed(self.pos_embed.shape[-1], self.grid_size)
        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))
        
        decoder_pos_embed = get_2d_sincos_pos_embed(self.decoder_pos_embed.shape[-1], self.grid_size)
        self.decoder_pos_embed.data.copy_(torch.from_numpy(decoder_pos_embed).float().unsqueeze(0))
        
        # Inicializar patch_embed como inicialización Xavier/Glorot
        w = self.patch_embed.weight.data
        torch.nn.init.xavier_uniform_(w.view([w.shape[0], -1]))
        torch.nn.init.normal_(self.mask_token, std=.02)

    def patchify(self, imgs):
        """Transforma (B, C, H, W) -> (B, N, patch_size**2 * C)"""
        p = self.patch_size
        h = w = self.img_size // p
        c = self.in_chans
        x = imgs.reshape(imgs.shape[0], c, h, p, w, p)
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(imgs.shape[0], h * w, p * p * c)
        return x

    def unpatchify(self, x):
        """Transforma (B, N, patch_size**2 * C) -> (B, C, H, W)"""
        p = self.patch_size
        h = w = self.img_size // p
        c = self.in_chans
        x = x.reshape(x.shape[0], h, w, p, p, c)
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(x.shape[0], c, h * p, w * p)
        return imgs

    def random_masking(self, x, mask_ratio):
        """
        Enmascara aleatoriamente el 75% de los parches.
        Retorna solo los parches visibles + la máscara binaria + índices de restauración.
        """
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        
        noise = torch.rand(N, L, device=x.device)  # Ruido uniforme
        ids_shuffle = torch.argsort(noise, dim=1)  # Ascendente
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        
        # Mantener solo los primeros len_keep parches visibles
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        
        # Generar máscara binaria: 0 es visible, 1 es enmascarado
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio):
        # Patch embedding: (B, C, H, W) -> (B, D, grid, grid) -> (B, N, D)
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        
        # Agregar posición espacial fija
        x = x + self.pos_embed
        
        # Enmascarar el 75%
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        
        # Pasar solo parches visibles por el Encoder
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        # Proyectar al espacio del Decoder
        x = self.decoder_embed(x)
        
        # Reinsertar tokens de máscara en las posiciones originales que faltan
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] - x.shape[1], 1)
        x_ = torch.cat([x, mask_tokens], dim=1)
        x = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2]))
        
        # Agregar posición en el Decoder
        x = x + self.decoder_pos_embed
        
        # Procesar con bloques del Decoder
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        
        # Predecir los píxeles multiespectrales reconstruidos
        x = self.decoder_pred(x)
        return x

    def forward_loss(self, imgs, pred, mask):
        """
        Calcula la pérdida espectral (MSE Loss) calculada sobre los parches ocultos.
        """
        target = self.patchify(imgs)
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)  # Error medio por parche
        
        # Evaluar la pérdida sobre los parches enmascarados
        loss = (loss * mask).sum() / mask.sum()
        return loss

    def forward(self, imgs, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask

## 6. 🧪 Prueba Rápida de Sanidad (Sanity Check)
Verificamos que el flujo de tensores, la reducción del 75% en el Encoder y la reconstrucción en el Decoder funcionen sin errores de dimensiones en GPU L4.

In [ ]:
# Instanciar modelo AgroMAE
model = AgroMAE(
    img_size=64, 
    patch_size=8, 
    in_chans=13,
    embed_dim=256, 
    depth=6, 
    num_heads=8,
    decoder_embed_dim=128, 
    decoder_depth=3, 
    decoder_num_heads=4,
    mask_ratio=0.75
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"⭐ Modelo AgroMAE instanciado con éxito. Total de parámetros entrenables: {total_params:,}")

# Simular un batch de prueba (4 imágenes, 13 bandas, 64x64)
dummy_input = torch.rand(4, 13, 64, 64).to(device)
with torch.no_grad():
    loss, pred, mask = model(dummy_input)
    reconstructed_imgs = model.unpatchify(pred)

print(f"✅ Sanity Check Exitoso:")
print(f"   - Entrada: {dummy_input.shape}")
print(f"   - Loss de reconstrucción inicial: {loss.item():.4f}")
print(f"   - Salida Predicha: {pred.shape}")
print(f"   - Imagen Reconstruida: {reconstructed_imgs.shape}")

## 7. 🚀 Bucle de Pre-Entrenamiento Auto-Supervisado con Checkpointing Continuo

In [ ]:
# Hiperparámetros de Entrenamiento
TOTAL_EPOCHS = 35
BASE_LR = 1.5e-4
WEIGHT_DECAY = 0.05
SAVE_EVERY_STEPS = 200  # Checkpoint intra-epoch cada 200 batches para Colab

# Optimizador AdamW con decaimiento de peso para regularización
optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)

# Scheduler de aprendizaje: Cosine Annealing con Warmup suave
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)

# GradScaler para Precisión Mixta (AMP en GPU L4)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

# Gestor de Checkpoints conectado al Google Drive
ckpt_manager = CheckpointManager(checkpoint_dir=CHECKPOINT_DIR, max_to_keep=2)

# Intentar reanudar automáticamente si Colab se desconectó previamente
start_epoch, global_step, best_val_loss = ckpt_manager.load_latest(
    model=model, 
    optimizer=optimizer, 
    scheduler=scheduler, 
    scaler=scaler, 
    device=device
)

### Ejecución del Bucle de Entrenamiento

In [ ]:
def validate(model, dataloader, device):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
                loss, _, _ = model(imgs)
            val_loss += loss.item()
    return val_loss / len(dataloader)

print(f"🏁 Iniciando entrenamiento desde Época {start_epoch} hasta {TOTAL_EPOCHS}...")

train_losses = []
val_losses = []

for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    epoch_loss = 0.0
    start_time = time.time()
    
    for batch_idx, (imgs, _) in enumerate(train_loader):
        imgs = imgs.to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
            loss, _, _ = model(imgs)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        global_step += 1
        
        # Checkpoint Intra-Epoch (cada N pasos)
        if global_step % SAVE_EVERY_STEPS == 0:
            ckpt_manager.save(
                epoch=epoch, 
                global_step=global_step, 
                model=model, 
                optimizer=optimizer, 
                scheduler=scheduler, 
                scaler=scaler, 
                loss=loss.item()
            )
            
    scheduler.step()
    train_loss = epoch_loss / len(train_loader)
    val_loss = validate(model, val_loader, device)
    elapsed = time.time() - start_time
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        
    print(f"Epoch [{epoch+1:02d}/{TOTAL_EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | Tiempo: {elapsed:.1f}s")
    
    # Guardar Checkpoint al final de cada época en Drive
    ckpt_manager.save(
        epoch=epoch, 
        global_step=global_step, 
        model=model, 
        optimizer=optimizer, 
        scheduler=scheduler, 
        scaler=scaler, 
        loss=val_loss, 
        is_best=is_best
    )

## 8. 🎨 Visualización de Resultados: Reconstrucción Espectral
Esta celda genera la comparación visual para el informe y diapositivas:
1. **Imagen Original** (Color Real RGB: Bandas B04, B03, B02).
2. **Entrada Enmascarada al 75%** (Solo el 25% visible).
3. **Reconstrucción AgroMAE** (Completada por el modelo).
4. **Falso Color Infrarrojo Vegetación (RGB 8-4-3)** (Banda NIR B08 en Rojo, B04 en Verde, B03 en Azul).

In [ ]:
def visualize_reconstruction(model, dataloader, device, num_samples=3):
    model.eval()
    imgs, _ = next(iter(dataloader))
    imgs = imgs[:num_samples].to(device)
    
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
            _, pred, mask = model(imgs)
            
        # Reconstruir imagen
        pred_imgs = model.unpatchify(pred)
        
        # Crear máscara visual sobre los parches
        mask_expanded = mask.unsqueeze(-1).repeat(1, 1, model.patch_size ** 2 * model.in_chans)
        mask_imgs = model.unpatchify(mask_expanded)
        masked_inputs = imgs * (1.0 - mask_imgs)
        
    imgs = imgs.cpu().float().numpy()
    masked_inputs = masked_inputs.cpu().float().numpy()
    pred_imgs = pred_imgs.cpu().float().numpy()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    if num_samples == 1:
        axes = np.expand_dims(axes, 0)
        
    # Índices de bandas de Sentinel-2:
    # RGB Natural: B04 (Rojo, idx 3), B03 (Verde, idx 2), B02 (Azul, idx 1)
    # Falso Color NIR: B08 (NIR, idx 7), B04 (Rojo, idx 3), B03 (Verde, idx 2)
    
    for i in range(num_samples):
        # RGB Original
        orig_rgb = np.clip(np.stack([imgs[i, 3], imgs[i, 2], imgs[i, 1]], axis=-1) * 3.5, 0, 1)
        axes[i, 0].imshow(orig_rgb)
        axes[i, 0].set_title("Original (RGB Sentinel-2)")
        axes[i, 0].axis('off')
        
        # Entrada Enmascarada al 75%
        masked_rgb = np.clip(np.stack([masked_inputs[i, 3], masked_inputs[i, 2], masked_inputs[i, 1]], axis=-1) * 3.5, 0, 1)
        axes[i, 1].imshow(masked_rgb)
        axes[i, 1].set_title("Enmascarado (75% Oculto)")
        axes[i, 1].axis('off')
        
        # Reconstrucción AgroMAE
        pred_rgb = np.clip(np.stack([pred_imgs[i, 3], pred_imgs[i, 2], pred_imgs[i, 1]], axis=-1) * 3.5, 0, 1)
        axes[i, 2].imshow(pred_rgb)
        axes[i, 2].set_title("Reconstrucción AgroMAE")
        axes[i, 2].axis('off')
        
        # Falso Color Infrarrojo (NIR 8-4-3)
        pred_nir = np.clip(np.stack([pred_imgs[i, 7], pred_imgs[i, 3], pred_imgs[i, 2]], axis=-1) * 3.0, 0, 1)
        axes[i, 3].imshow(pred_nir)
        axes[i, 3].set_title("Reconstrucción Falso Color NIR")
        axes[i, 3].axis('off')
        
    plt.tight_layout()
    plt.show()

visualize_reconstruction(model, val_loader, device, num_samples=3)

## 9. 🏆 Preparación para Fase 2: Cabezal de Clasificación Downstream (Fine-Tuning)
Descartamos el *Decoder* y acoplamos una capa de clasificación densa al *Encoder* pre-entrenado para clasificar los 10 usos de suelo de EuroSAT.

In [ ]:
class AgroMAEClassifier(nn.Module):
    """
    Clasificador Downstream que transfiere los pesos del Encoder de AgroMAE.
    """
    def __init__(self, pretrained_mae, num_classes=10):
        super().__init__()
        # Extraer encoder y posiciones espaciales
        self.patch_embed = pretrained_mae.patch_embed
        self.pos_embed = pretrained_mae.pos_embed
        self.blocks = pretrained_mae.blocks
        self.norm = pretrained_mae.norm
        
        # Cabezal de Clasificación (Linear Probe / Fine-Tuning)
        embed_dim = pretrained_mae.pos_embed.shape[-1]
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # En downstream, la imagen entra 100% visible (sin enmascarar)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        
        # Global Average Pooling espacial sobre los parches
        x = x.mean(dim=1)
        logits = self.head(x)
        return logits

# Instanciar Clasificador Downstream
classifier = AgroMAEClassifier(pretrained_mae=model, num_classes=10).to(device)
print("✅ Modelo Clasificador Downstream preparado exitosamente para el Gran Experimento de Comparación.")